In [1]:
import requests as req
from bs4 import BeautifulSoup
import time
import os
from dotenv import load_dotenv
from PIL import Image
from io import BytesIO

In [2]:
load_dotenv()

True

In [3]:
m_url = "https://www.mycourseville.com/?q=onlinecourse/quiz/"
urls_id = [
    1286311,
    1251971,
    1284559,
    1284754,
    1284805,
    1284852,
]

In [4]:
htmls = []
my_cookie = os.environ["TOKEN_COOKIE"]

In [5]:
header = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Referer": "https://www.mycourseville.com/?q=onlinecourse/course/51604",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "Upgrade-Insecure-Requests": "1",
    "Cookie": my_cookie
}

In [6]:
for id in urls_id:
    response = req.post(url=f"{m_url}{id}", headers=header)
    time.sleep(1)
    if response.status_code == 200:
        html = response.text
        htmls.append(html)  
    else:
        print(f"id: {id} course error") 

In [7]:
question_each_course = []
answer_each_course = []
question_image_eachcourse = []
name_each_image_eachcourse = []
questions_code = []

In [ ]:
for html in htmls:
    soup = BeautifulSoup(html, "lxml")
    title = soup.select_one("a[href*='?q=onlinecourse/course/'][title='Go to the learning path of the course']").text
    questions = soup.select("div.cvocp-quiz-item")
    question_course = []
    question_image_course = []
    question_code = []
    answer_course = []
    question_name_image = []
    for i, question in enumerate(questions):
        questions_part = question.select_one("div[data-part='question-body']")
        responses_part = question.select("div[data-part='choice-item']")
        have_imgs_question = questions_part.select("p")
        have_code_question = questions_part.select("pre")
        imgs_question = []
        code_question = []
        name_question = []
        q_parts = [] 

        if len(have_imgs_question) != 0:
            for p in have_imgs_question:
                img = p.select_one("img")
                if img:
                    url = img.get("src")
                    if not url.startswith("http"):
                        url = "https://www.mycourseville.com" + url
                    res = req.get(url) 
                    image = Image.open(BytesIO(res.content))
                    imgs_question.append(image)
                else:
                    text = p.text.strip()
                    if not text:
                        continue
                    is_label = len(text) <= 3 or (
                        (text.startswith("รูป") or text.startswith("ภาพ") or text.startswith("Figure")) 
                        and len(text) <= 20
                    )
                    
                    if is_label:
                        name_question.append(text)
                    else:
                        q_parts.append(text)
            q = "\n".join(q_parts)
        else:
            q = questions_part.text.strip()
        if len(have_code_question) != 0:
            for c in have_code_question:
                code_question.append(c.text.strip())

        question_course.append(q)
        question_image_course.append(imgs_question)
        question_code.append(code_question)
        question_name_image.append(name_question)

        choics = []
        for choice in responses_part:
            ans = choice.select_one("span[data-part='choice-content']")
            choics.append(ans.text.strip())
        
        answer_course.append(choics)

    question_each_course.append(question_course)
    answer_each_course.append(answer_course)
    question_image_eachcourse.append(question_image_course)
    questions_code.append(question_code)
    name_each_image_eachcourse.append(question_name_image)

In [10]:
question_each_course[0]

['ตัวเลือกใดแสดงแนวคิดของการทำงานโดยคอมพิวเตอร์เชิงไฟฟ้า',
 'ทรานซิสเตอร์ถูกใช้อย่างไรในระบบคอมพิวเตอร์',
 'กำหนดข้อมูลเลขฐานสอง ขนาด 8 หลักเป็น 10010000 ตัวเลือกใดถูกต้อง',
 '',
 'หากกำหนด A = 0100 และ B = 0010 ตัวเลือกใดเป็นผลลัพธ์ของ A AND B',
 'จากตัวเลือกต่อไปนี้ หน่วยความจำแบบใดทำงานได้เร็วที่สุด',
 'ตัวเลือกใดเป็นส่วนคำนวณของคอมพิวเตอร์',
 'คอมพิวเตอร์ประกอบด้วยฮาร์ดแวร์ชิ้นต่างๆ จำนวนมาก การทำงานร่วมกันของอุปกรณ์ต่างๆ เหล่านี้เกิดขึ้นได้อย่างไร',
 'ภาษาแอสเซมบลีคืออะไร',
 'การมี core หรือแกนประมวลผล มากขึ้น มีประโยชน์อย่างไร',
 'ตัวเลือกใดเป็นหน้าที่ของระบบปฏิบัติการ',
 'เหตุใดการสร้างไฟล์ เช่น ไฟล์ภาพ JPG บนโทรศัพท์เคลื่อนที่ สามารถนำไปเปิดในเครื่องอื่นๆ เช่น คอมพิวเตอร์ตั้งโต๊ะ ได้เช่นกัน แม้เครื่องนั้นจะมีสถาปัตยกรรมที่ต่างกัน',
 'ตัวเลือกใดถูกต้องเกี่ยวกับการเก็บข้อมูลภาพ',
 'หากข้อมูลทุกรูปแบบในคอมพิวเตอร์ปัจจุบันเก็บโดยใช้เลขฐานสองทั้งหมด ตัวเลือกใดต่อไปนี้ถูกต้อง',
 '']

In [12]:
question_image_eachcourse

[[[], [], [], [], [], [], [], [], [], [], [], [], [], [], []],
 [[], [], [], [], [], [], [], [], [], []],
 [[],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  []],
 [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []],
 [[],
  [],
  [],
  [],
  [],
  [],
  [<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1232x757>],
  [<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1170x717>,
   <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1084x729>,
   <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1101x721>,
   <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1108x742>,
   <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1088x748>],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  [],
  []],
 [[],
  [],
  [<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1930x1310>],
  [],
  [<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1666x832>],
  [],
  [],
